# PART 3

In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Data generation

First of all, we will implement the methods to generate the dataset.

In [41]:
def generate_data(m, n):
    X = np.random.choice([-1, 1], size=(m, n))
    y = X[:, 0].copy()
    
    return X, y

## Algorithm Implementation

#### 1. Perceptron

In [42]:
def perceptron(X_train, y_train, X_test, epochs=10):
    m = X_train.shape[0]
    n = X_train.shape[1]
    W = np.zeros(n)
    # train 
    for _ in range(epochs):
        for i in range(m):
            Xi = X_train[i]
            yi = y_train[i]
            yi_pred = W@Xi # predict
            if yi*yi_pred <= 0: # mistake
                W += yi*Xi
    # test
    y_pred = np.sign(X_test@W)
    y_pred[y_pred == 0] = -1 # if X@w=0, predict with -1
    return y_pred

#### 2. Winnow

In [43]:
def winnow(X_train, y_train, X_test, epochs=10):
    m = X_train.shape[0]
    n = X_train.shape[1]
    W = np.ones(n)
    # transform data to 0/1 (winnow works with 0/1 labels)
    X_train_bin = (X_train + 1)/2
    y_train_bin = (y_train + 1)/2
    X_test_bin = (X_test + 1)/2
    for _ in range(epochs):
        for i in range(m):
            Xi = X_train_bin[i]
            yi = y_train_bin[i]
            yi_pred = np.float64(W@Xi >= n)
            if yi != yi_pred: # mistake
                if yi == 1:
                    W *= 2**Xi
                else:
                    W *= 0.5**Xi
    # test
    y_pred_bin = np.float64(X_test_bin@W >= n)
    y_pred = y_pred_bin*2 - 1 # convert 0 to -1
    return y_pred

#### 3. Least Squares

In [44]:
def least_squares(X_train, y_train, X_test):
    # as m is going to be lower than n in most cases, X^T*X won't be invertible
    # therefore, we compute W = pinv(X)*y, as an approximation to X*W=y
    W = np.linalg.pinv(X_train)@y_train
    # predict
    y_pred = np.sign(X_test@W)
    return y_pred

#### 4. 1-Nearest Neighbors

In [ ]:
def one_nearest_neighbours(X_train, y_train, X_test):
    m = X_test.shape[0]
    y_pred = np.zeros(m)
    for i in range(m):
        distances = np.linalg.norm(X_test[i]-X_train, axis=1)
        neighbours = np.where(distances==distances.min())[0] # indexes of nearest neghbours (there may be more than one)
        chosen_neighbour = np.random.choice(neighbours) # select one random neighbour from the nearest ones
        y_pred[i] = y_train[chosen_neighbour]
    return y_pred

## Estimate sample complexity

In [65]:
def empirical_error(y, y_pred):
    error = np.sum(y != y_pred)/(len(y))
    return error

def estimate_sample_complexity(algorithm, n_list, m_train=1000, m_test=1000, error_bound=0.1, n_trials=10):
    """_summary_

    Args:
        algorithm (_type_): _description_
        n_list (_type_): list of onput dimensions to measure sample complexity on
        m_train (int, optional): maximum size of train dataset. Defaults to 10000.
        m_test (int, optional): test dataset size. Defaults to 10000.
        error_bound (float, optional): generalisation error bound used in the definition of sample complexity. Defaults to 0.1 (10%).
    """
    sample_complexity = dict() # dict to store sample complexities
    for n in n_list: # iterate through input dimensions
        m_list = np.zeros(n_trials)
        for i in range(n_trials): # trial n times
            # generate train and test data
            X_train, y_train = generate_data(m=m_train, n=n)
            X_test, y_test = generate_data(m=m_test, n=n)
            found=False # set to false before each iteration
            for m in range(1, m_train): # iterate though sample sizes
                # m first train datapoints
                Xm = X_train[:m] 
                ym = y_train[:m]
                # predictions
                y_pred = algorithm(Xm, ym, X_test) 
                # error rate
                error_rate = empirical_error(y_test, y_pred) 
                if error_rate <= error_bound: # error rate lower than bound 
                    m_list[i] = m
                    found = True
                    break
            if not found: 
                m_list[i] = m_train
        sample_complexity[n] = (m_list.mean(), m_list.std()) # add mean and std of sample complexity for the input dimension
    return sample_complexity # return dictionary

def plot_sample_complexity(complexity_dict, algorithm_name="Algorithm"):
    """
    Plots the sample complexity (m) vs dimension (n) with error bars.
    Matches the style of the provided reference image.
    
    Args:
        complexity_dict (dict): Dictionary where key=n, value=(mean_m, std_m)
        algorithm_name (str): Label for the legend (e.g., 'Least Squares')
    """
    # Dictionaries are unordered, so we must sort keys to draw the line correctly
    sorted_n = sorted(complexity_dict.keys())
    
    # Extract means and standard deviations in the sorted order
    means = [complexity_dict[n][0] for n in sorted_n]
    stds  = [complexity_dict[n][1] for n in sorted_n]
    
    # fmt='-o': Connect points with lines (-), use circle markers (o)
    # capsize=4: Adds the horizontal caps to the error bars (like the image)
    plt.errorbar(sorted_n, means, yerr=stds, 
                 fmt='-o', capsize=4, label=algorithm_name)

    # 4. Labeling and Styling
    plt.xlabel('Dimension (n)')
    plt.ylabel('Sample Complexity (m)')
    plt.title('Sample Complexity vs. Dimension')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()